In [ ]:
# Week 3 Day 5: RandomizedSearchCV and Efficient Hyperparameter Tuning

## Today's Goal

- Review how GridSearchCV searches hyperparameters
- Understand why exhaustive search becomes expensive
- Understand the basic idea of RandomizedSearchCV
- Understand the role of n_iter
- Compare search-space size with the number of evaluated configurations
- Use RandomizedSearchCV with a sklearn Pipeline
- Interpret randomized search results carefully

## Expected Output

By the end of today, I should be able to:

1. Explain the difference between GridSearchCV and RandomizedSearchCV
2. Calculate the size of a hyperparameter search space
3. Explain what n_iter controls
4. Run RandomizedSearchCV with cross-validation
5. Explain the efficiency-performance trade-off of randomized search

In [6]:
## Data Preparation and Pipeline
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Load the same Banknote Authentication dataset
banknote = fetch_openml(
    data_id=1462,
    as_frame=True
)

X = banknote.data
y = banknote.target

# Hold out the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build preprocessing + model pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

print("Development set:", X_dev.shape)
print("Final test set:", X_test.shape)

Development set: (1097, 4)
Final test set: (275, 4)


In [7]:
## Data Preparation and Pipeline
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Load the same Banknote Authentication dataset
banknote = fetch_openml(
    data_id=1462,
    as_frame=True
)
X = banknote.data
y = banknote.target

# Hold out the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build preprocessing + model pipeline
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

print("Development set:", X_dev.shape)
print("Final test set:", X_test.shape)

Development set: (1097, 4)
Final test set: (275, 4)


In [3]:
## Randomized Search Space
param_distributions = {
    "knn__n_neighbors": list(range(1, 31)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

print("Number of possible configurations:")
print(30 * 2 * 2)

Number of possible configurations:
120


In [8]:
## RandomizedSearchCV Setup
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Define 5-fold stratified CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=knn_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [9]:
random_search.fit(X_dev, y_dev)

print("Best parameters:")
print(random_search.best_params_)

print("\nBest mean CV accuracy:")
print(random_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters:
{'knn__weights': 'distance', 'knn__p': 2, 'knn__n_neighbors': 12}

Best mean CV accuracy:
0.9990909090909093


In [ ]:
import pandas as pd

random_results = pd.DataFrame(random_search.cv_results_)

random_results_table = random_results[
    [
        "param_knn__n_neighbors",
        "param_knn__weights",
        "param_knn__p",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

random_results_table = random_results_table.sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False]
)

print(random_results_table.to_string(index=False))

In [10]:
import pandas as pd

random_results = pd.DataFrame(random_search.cv_results_)

random_results_table = random_results[
    [
        "param_knn__n_neighbors",
        "param_knn__weights",
        "param_knn__p",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

random_results_table = random_results_table.sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False]
)

print(random_results_table.to_string(index=False))

 param_knn__n_neighbors param_knn__weights  param_knn__p  mean_test_score  std_test_score  rank_test_score
                     12           distance             2         0.999091        0.001818                1
                     14           distance             2         0.999091        0.001818                1
                     19           distance             1         0.999091        0.001818                1
                     27           distance             2         0.999091        0.001818                1
                      2            uniform             1         0.998182        0.002227                5
                      7            uniform             2         0.998182        0.002227                5
                      3            uniform             2         0.998182        0.002227                5
                      5            uniform             2         0.998182        0.002227                5
                     12            un

In [ ]:
## Final Test Evaluation
from sklearn.metrics import accuracy_score

# RandomizedSearchCV already refitted the selected model
# on all development data
best_random_model = random_search.best_estimator_

# Final prediction on untouched test data
y_test_pred = best_random_model.predict(X_test)

final_test_accuracy = accuracy_score(y_test, y_test_pred)

print("Best sampled parameters:")
print(random_search.best_params_)

print("\nBest mean CV accuracy:")
print(random_search.best_score_)

print("\nFinal test accuracy:")
print(final_test_accuracy)

In [11]:
## Final Test Evaluation
from sklearn.metrics import accuracy_score

# RandomizedSearchCV already refitted the selected model
# on all development data
best_random_model = random_search.best_estimator_

# Final prediction on untouched test data
y_test_pred = best_random_model.predict(X_test)

final_test_accuracy = accuracy_score(y_test, y_test_pred)

print("Best sampled parameters:")
print(random_search.best_params_)

print("\nBest mean CV accuracy:")
print(random_search.best_score_)

print("\nFinal test accuracy:")
print(final_test_accuracy)

Best sampled parameters:
{'knn__weights': 'distance', 'knn__p': 2, 'knn__n_neighbors': 12}

Best mean CV accuracy:
0.9990909090909093

Final test accuracy:
1.0


In [ ]:
# Reflection

1. What is the main difference between GridSearchCV and RandomizedSearchCV?
GridSearchCV exhaustively searched all configurations with k-fold CV while RandomizedSearchCV randomly searched particular sample 
configurations rather than all configurations.
2. Why can RandomizedSearchCV be useful even if it does not evaluate every hyperparameter configuration?
Because RandomizedSearchCV randomly selects particular sample configurations so it most likely evalautes and finds the configuration with 
well performance which probably is tied with the best configuration.
3. Why should we avoid repeatedly comparing different models on the same final test set?
Because the final test set should be independent and isolated. If we repeatedly campare different models in this stage, the final test set
will be data contamination and the results will be unreliable.